## Setup

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "1.0-POC"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


## Datasets

### In Conversation Corpus (ICC) 

#### Variant - No Labels

This variant of the ICC does not include any conversation start or end or 
speaker labels. 

In [7]:
from LLMDialogue.datasets import load
from LLMDialogue.datasets.ICC import objectify as ICCObjectify, ICCDatasetSplits

In [8]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
print(dset)

{
    "date_created": "21_05_2024__12_13_53",
    "dset_name": "ICC",
    "variant_name": "no_labels",
    "version": "0.0.1",
    "data_dir": null,
    "data_files": {
        "train": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/train",
        "test": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/test"
    },
    "force_regenerate": false,
    "config_id": "6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019",
    "cache_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019",
    "archive_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019/archive",
    "compress_archive": true,
    "seed":

In [9]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names


['train', 'test']

In [10]:
dset_splits.data_df_map['train']

,conversationName,conversationID,turnNumber,utterance
0,2018-02-04-session-1_nogaps_step1_GPT,0,0,all right
1,2018-02-04-session-1_nogaps_step1_GPT,0,1,hel
2,2018-02-04-session-1_nogaps_step1_GPT,0,2,hela hela this is go so weird to get used to
3,2018-02-04-session-1_nogaps_step1_GPT,0,3,i know i feel kinda
4,2018-02-04-session-1_nogaps_step1_GPT,0,4,all right um when's your first softball game
...,...,...,...,...
16825,2018-10-20-session-3_nogaps_step1_GPT,27,453,i'm like no high school junior or senior looke...
16826,2018-10-20-session-3_nogaps_step1_GPT,27,454,you<ts> i know exactly
16827,2018-10-20-session-3_nogaps_step1_GPT,27,455,not at my school
16828,2018-10-20-session-3_nogaps_step1_GPT,27,456,like it the the freshmen look like literal chi...


In [11]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'train': './outputs/1.0-POC/ICC/no_labels/train.csv',
 'test': './outputs/1.0-POC/ICC/no_labels/test.csv'}

#### Variant - Special Labels

In [12]:
dset = load(
    dset_name="ICC", 
    config_name="labels"
)
print(dset)

{
    "date_created": "21_05_2024__12_13_53",
    "dset_name": "ICC",
    "variant_name": "labels",
    "version": "0.0.1",
    "data_dir": null,
    "data_files": {
        "train": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/train",
        "test": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/test"
    },
    "force_regenerate": false,
    "config_id": "16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c",
    "cache_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c",
    "archive_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c/archive",
    "compress_archive": true,
    "seed": 42

In [13]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names

['train', 'test']

In [14]:
dset_splits.data_df_map['train']

,conversationName,conversationID,turnNumber,utterance
0,2018-02-04-session-1_nogaps_step1_GPT,0,0,<START>
1,2018-02-04-session-1_nogaps_step1_GPT,0,1,<SP1> all right <SP1>
2,2018-02-04-session-1_nogaps_step1_GPT,0,2,<SP2> hel <SP2>
3,2018-02-04-session-1_nogaps_step1_GPT,0,3,<SP1> hela hela this is go so weird to get use...
4,2018-02-04-session-1_nogaps_step1_GPT,0,4,<SP2> i know i feel kinda <SP2>
...,...,...,...,...
21154,2018-10-20-session-3_nogaps_step1_GPT,27,567,<SP1> not at my school <SP1>
21155,2018-10-20-session-3_nogaps_step1_GPT,27,568,<SP2> like it the the freshmen look like liter...
21156,2018-10-20-session-3_nogaps_step1_GPT,27,569,<SP1> yeah the freshmen look like children and...
21157,2018-10-20-session-3_nogaps_step1_GPT,27,570,<SP1> like men <SP1>


In [15]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'train': './outputs/1.0-POC/ICC/labels/train.csv',
 'test': './outputs/1.0-POC/ICC/labels/test.csv'}

### Speaker Identity

#### Variant - No Labels

In [16]:
from LLMDialogue.datasets.speakerIdentity import SpeakerIdentityDataSplits, objectify as SIObjectify

In [17]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [18]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

['test']

In [19]:
dset_splits.data_df_map['test']

,conversationName,conversationID,turnNumber,utterance
0,30b,0,0,<START>
1,30b,0,1,<SP1> i haven't seen the keys anywhere <SP1>
2,30b,0,2,<SP2> have you <SP2>
3,30b,0,3,<END>
4,101bf,1,0,<START>
...,...,...,...,...
2995,18bd,748,3,<END>
2996,4bf,749,0,<START>
2997,4bf,749,1,<SP1> it's great that you could cover this shi...
2998,4bf,749,2,<SP1> for real <SP1>


In [20]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'test': './outputs/1.0-POC/speakerIdentity/labels/test.csv'}

#### Labels

In [21]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [22]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

['test']

In [23]:
dset_splits.data_df_map['test']

,conversationName,conversationID,turnNumber,utterance
0,30b,0,0,<START>
1,30b,0,1,<SP1> i haven't seen the keys anywhere <SP1>
2,30b,0,2,<SP2> have you <SP2>
3,30b,0,3,<END>
4,101bf,1,0,<START>
...,...,...,...,...
2995,18bd,748,3,<END>
2996,4bf,749,0,<START>
2997,4bf,749,1,<SP1> it's great that you could cover this shi...
2998,4bf,749,2,<SP1> for real <SP1>


In [24]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)

{'test': './outputs/1.0-POC/speakerIdentity/labels/test.csv'}

### Data Modules

These are required to be able to train the model 

#### HuggingFace Language Modeling DM

In [25]:
from LLMDialogue.dm.HFdm import HFLanguageModelingDM
from transformers import AutoTokenizer

In [26]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
dset_splits = ICCObjectify(dset)
df_map = dict()
for name, df in dset_splits.data_df_map.items():
    remove_labels = [c for c in df.columns if c != 'utterance']
    # df.drop(labels=remove_labels)
    df = df.drop(remove_labels,axis=1)
    df = df.rename(columns={"utterance": "text"})
    if name == "test":
        name = "validation"
    df_map[name] = df
df_map.keys()

dict_keys(['train', 'validation'])

In [27]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [28]:
dm = HFLanguageModelingDM(
    df_map, 
    tokenizer,
    f"{OUTPUT_DIR}/dm"
)
dm.prepare_data()
dm.setup(stage="fit")

AttributeError: 'dict' object has no attribute 'map'

In [ ]:


from datasets import load_dataset
from datasets import Dataset


In [ ]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
dset_splits = ICCObjectify(dset)

In [ ]:
df = dset_splits.data_df_map['train']
remove_labels = [c for c in df.columns if c != 'utterance' and c != 'conversationID']
# df.drop(labels=remove_labels)
df = df.drop(remove_labels,axis=1)

In [ ]:
dataset_train = Dataset.from_pandas(df)
dataset_test = Dataset.from_pandas(df)

In [ ]:
from datasets import load_dataset, load_from_disk, concatenate_datasets, DatasetDict

In [ ]:
concatenate_datasets([dataset_train,dataset_test])

In [ ]:
dataset = DatasetDict({
    "train" : dataset_train, 
    "test" : dataset_test
})

In [ ]:
def _group_by(d, col, join):
        """from: https://github.com/huggingface/datasets/issues/3644"""
        # Get the indices of each group
        groups = {key: [] for key in d.unique(col)}

        def create_groups_indices(key, i):
            groups[key].append(i)

        d.map(create_groups_indices, with_indices=True, input_columns=col)
        # Get one dataset object per group
        groups = {key: d.select(indices) for key, indices in groups.items()}
        # Apply join function
        groups = {
            key: dataset_group.map(
                join,
                batched=True,
                batch_size=len(dataset_group),
                remove_columns=d.column_names,
            )
            for key, dataset_group in groups.items()
        }
        # Return concatenation of all the joined groups
        return concatenate_datasets(list(groups.values()))

In [ ]:
def _join(batch):
        """Simply return the Utterance values of the joined batch."""
        return {'utterance': [batch['utterance']]}

In [ ]:
for split in ("train", "test"):
    dataset[split] = _group_by(
        dataset[split], 'conversationID', _join
    )

In [ ]:
dataset['train']['utterance']

In [ ]:
from LLMDialogue.dm.ICCdm import ICCHFLMDM
from LLMDialogue.dm.HFdm import HFLanguageModelingDM

In [ ]:
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=tokenizer, 
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

In [ ]:
dm.val_dataloader()

In [29]:
from LLMDialogue.dm.ICCdm import ICCSpokenLanguageModelingDM
from LLMDialogue.tokenizers import SpokenDialogueTokenizer

In [30]:
tokenizer = SpokenDialogueTokenizer()

Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers


In [31]:
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/743 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/802 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/591 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/859 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

In [32]:
dm.prepare_data()
dm.setup('fit')

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4163 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1917 [00:00<?, ? examples/s]

In [36]:
next(iter(dm.train_dataloader()))

/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/src/LLMDialogue/dm/spokenLanguageDM.py:124: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ret[k] = torch.tensor(v)


{'input_ids': tensor([[ 1279,  9688,    29,  ...,   804,   379,   262],
        [ 7269,   475,  1312,  ...,  2777,    16,    29],
        [50257,  1279,  2777,  ...,   345,  4964,   262],
        ...,
        [ 1279,  2777,    17,  ...,    29,  1107,  1279],
        [ 2777,    17,    29,  ...,  3597,   465, 46793],
        [  319,   644,   339,  ...,  2777,    16,    29]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'speaker_ids': tensor([[50258, 50258, 50258,  ..., 50258, 50258, 50258],
        [50258, 50258, 50258,  ..., 50258, 50258, 50258],
        [50258, 50259, 50259,  ..., 50258, 50258, 50258],
        ...,
        [50258, 50258, 50258,  ..., 50259, 50259, 50259],
        [50259, 50259, 50259,  ..., 50259, 50259, 50259],
        [50259, 50259, 50259,  ..., 50259, 50259, 50259]])}